# Figure 2e — detections per cell across the z-stack

How many times the same cell is detected as you move through the stack. A cell
spans several optical sections, so 2D-per-slice segmentation counts it more than
once; this quantifies that duplication factor and is what stops the per-slice
counts in Fig 2d being read as cell numbers.

**Provenance — NEW CODE.** No plotting code for this panel survives in any source
tree, and `PORT_TRIAGE.md` did not list it. The inputs do survive: the spheroid
detection step wrote `z_duplicate_analysis_cell.csv` per plate, and those six files
are committed under `analysis/3_SupplFigure2/data/spher-colo52/`. This notebook
plots `avg_detections_per_cell` from them. The quantity is taken straight from the
column; the styling is inferred from the published panel and is not a port.


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import metadata
from utils.panels import save_panel

import glob, os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# One CSV per plate, written by the spheroid-detection step and committed with the
# Suppl 2 inputs. Barcode comes from the plate folder name (PB000137_spher-colo52_...).
DUP_GLOB = str(ROOT / "analysis" / "3_SupplFigure2" / "data" / "spher-colo52"
               / "*" / "segmentation_output" / "z_duplicate_analysis_cell.csv")

frames = []
for f in sorted(glob.glob(DUP_GLOB)):
    barcode = os.path.basename(os.path.dirname(os.path.dirname(f))).split("_")[0]
    frames.append(pd.read_csv(f).assign(barcode=barcode))

dup = pd.concat(frames, ignore_index=True)
# wells are "B08_s1"; the metadata keys on the bare well id
dup["well_id"] = dup["well"].str.split("_").str[0]
print(f"{len(dup)} wells across {dup.barcode.nunique()} plates")
dup.head()

In [ ]:
# Join the cell line on — plates 139/140 carry both lines, so it cannot be taken
# from the plate name alone.
meta = pd.read_csv(metadata("spher_colo52-metadata.csv", "exp1_main"))
meta = meta[["barcode", "well_id", "cell_line", "pert_type"]].drop_duplicates()

dup = dup.merge(meta, on=["barcode", "well_id"], how="left")
assert dup["cell_line"].notna().all(), "some wells did not match the metadata"

print(dup.groupby("cell_line")["avg_detections_per_cell"]
         .describe()[["count", "mean", "50%"]].to_string())
print("\nexpected z-span per cell (slices):", sorted(dup.expected_z_span_slices.unique()))

In [ ]:
HUE_ORDER = ["HT29", "HCT116"]     # same ordering and ramp as Fig 2d

fig, ax = plt.subplots(figsize=(2.6, 5))
sns.boxplot(
    data=dup, x=[""] * len(dup), y="avg_detections_per_cell",
    hue="cell_line", hue_order=HUE_ORDER, palette="dark:grey",
    width=0.5, fill=True, legend=False, ax=ax,
    # only the outliers are drawn as dots, as in the paper — not every well
    showfliers=True, flierprops=dict(marker="o", markersize=3, alpha=0.55,
                                     markerfacecolor="0.35", markeredgecolor="none"),
)

ax.set_ylabel("Detections per cell (Z-stack)", fontsize=12)
ax.set_xlabel("duplicate\nestimate", fontsize=12)
ax.set_ylim(-0.2, 5.2)
sns.despine()

save_panel(
    fig, "Fig2e",
    data=dup[["barcode", "well", "cell_line", "n_unique_cells", "n_total_detections",
              "avg_detections_per_cell", "median_detections", "expected_z_span_slices"]],
    caption="Detections per cell through the z-stack, by cell line",
    notebook="analysis/3_Figure2/CellDetectionSanityCheck/3_DuplicateDetections.ipynb",
)
plt.show()